# 🚢 Логистическая регрессия с нуля на Titanic
## Функциональный подход + детальное логирование

В этом ноутбуке мы реализуем логистическую регрессию **без использования `sklearn`** для классификации выживших в датасете Titanic.  
Реализация построена в функциональном стиле, каждая функция сопровождается теоретическим блоком.  
Все шаги обучения, значения функции потерь, параметры и метрики записываются в файл `training_log.csv` для последующего анализа.

> 💡 **Как использовать:** Запускайте ячейки последовательно. После обучения откройте `training_log.csv` или воспользуйтесь функцией визуализации логов в конце.

In [8]:
import numpy as np
import pandas as pd
import csv
import os
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка отображения
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## 📊 Подготовка данных
Логистическая регрессия чувствительна к масштабу признаков и пропущенным значениям. Перед обучением необходимо:
1. Заполнить пропуски (медиана для `Age`, мода для `Embarked`)
2. Удалить неинформативные колонки (`Name`, `Ticket`, `Cabin`, `PassengerId`)
3. Закодировать категориальные признаки (`Sex`, `Embarked`)
4. **Масштабировать** признаки (`StandardScaler`), иначе градиентный спуск будет сходиться медленно или расходиться.

Математически масштабирование преобразует признаки к виду:
$$ x' = \frac{x - \mu}{\sigma} $$
где $\mu$ — среднее, $\sigma$ — стандартное отклонение.

In [1]:
def load_and_preprocess_titanic(test_size=0.2, random_state=42):
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    
    # Заполнение пропусков
    df['Age'].fillna(df['Age'].median(), inplace=True)
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
    
    # Удаление "шумных" признаков
    df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)
    
    # Кодирование категориальных признаков
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
    
    X = df.drop('Survived', axis=1).values
    y = df['Survived'].values
    
    # Разделение и масштабирование
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    return X_train, X_test, y_train, y_test, df.drop('Survived', axis=1).columns.tolist()

## 📈 Функция активации: Сигмоида
Логистическая регрессия предсказывает вероятность принадлежности к классу 1. Для этого линейная комбинация $z = \mathbf{w}^T\mathbf{x} + b$ пропускается через сигмоиду:

$$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

Свойства:
- $\sigma(z) \in (0, 1)$ при $z \in \mathbb{R}$
- $\sigma(0) = 0.5$
- При больших $|z|$ функция "насыщается", поэтому важна численная стабильность.

В коде используется `np.clip` для защиты от переполнения `exp()`.

In [2]:
def sigmoid(z):
    """Численно стабильная сигмоида."""
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

## 📉 Функция потерь: Binary Cross-Entropy (BCE)
Для бинарной классификации используется логарифмическая функция потерь:

$$ J(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right] $$

Где:
- $y^{(i)}$ — истинная метка (0 или 1)
- $\hat{y}^{(i)} = \sigma(\mathbf{w}^T\mathbf{x}^{(i)} + b)$ — предсказанная вероятность
- $m$ — количество примеров

Функция сильно штрафует уверенные, но неверные предсказания (например, $\hat{y}=0.99$ при $y=0$).

In [3]:
def compute_loss(y_true, y_pred):
    """Binary Cross-Entropy Loss."""
    m = len(y_true)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)  # защита от log(0)
    return -1/m * np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

## 🧮 Градиенты и L2-регуляризация
Градиентный спуск обновляет параметры в направлении, противоположном градиенту функции потерь:

$$ \frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} X^T (\hat{\mathbf{y}} - \mathbf{y}) + \frac{\lambda}{m} \mathbf{w} $$
$$ \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) $$

Второе слагаемое в первой формуле — **L2-регуляризация** (Ridge). Она добавляет штраф за большие веса, предотвращая переобучение. $\lambda$ — гиперпараметр силы регуляризации. Для смещения $b$ регуляризация обычно не применяется.

In [4]:
def compute_gradients(X, y, y_pred, w, lambda_reg=0.0):
    """Возвращает градиенты по весам и смещению с опциональной L2-регуляризацией."""
    m = len(y)
    error = y_pred - y
    dw = (1/m) * np.dot(X.T, error) + (lambda_reg/m) * w
    db = (1/m) * np.sum(error)
    return dw, db

## 🔄 Цикл обучения и логирование
На каждой эпохе:
1. Вычисляем предсказания: $\hat{y} = \sigma(X\mathbf{w} + b)$
2. Считаем градиенты и обновляем параметры: $\mathbf{w} := \mathbf{w} - \alpha \cdot \nabla_{\mathbf{w}} J$
3. Записываем в файл: эпоху, loss, learning rate, регуляризацию, смещение, статистику весов и accuracy на train.

Логирование в `training_log.csv` позволяет:
- Отслеживать сходимость
- Диагностировать колебания или расходимость (слишком большой `lr`)
- Визуализировать влияние регуляризации

In [5]:
LOG_FILE = "training_log.csv"
PARAMS_FILE = "model_params.json"

def init_log_file():
    """Создаёт/перезаписывает заголовок лога."""
    with open(LOG_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'loss', 'lr', 'lambda_reg', 'bias', 'w_mean', 'w_std', 'train_acc'])

def log_step(epoch, loss, lr, lam, b, w, train_acc):
    """Добавляет строку в CSV-лог."""
    with open(LOG_FILE, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, f"{loss:.6f}", f"{lr:.4f}", f"{lam:.4f}", 
                         f"{b:.6f}", f"{np.mean(w):.6f}", f"{np.std(w):.6f}", f"{train_acc:.4f}"])

def train_logistic_regression(X, y, lr=0.1, epochs=1000, lambda_reg=0.0, log_interval=50):
    m, n = X.shape
    w = np.zeros(n)
    b = 0.0
    loss_history = []
    
    init_log_file()
    print(f"🚀 Старт обучения | LR={lr} | Epochs={epochs} | Lambda={lambda_reg}")

    for epoch in range(epochs):
        z = np.dot(X, w) + b
        y_pred = sigmoid(z)
        
        dw, db = compute_gradients(X, y, y_pred, w, lambda_reg)
        w -= lr * dw
        b -= lr * db
        
        loss = compute_loss(y, y_pred)
        loss_history.append(loss)
        
        if epoch % log_interval == 0:
            train_acc = accuracy_score(y, (y_pred >= 0.5).astype(int))
            log_step(epoch, loss, lr, lambda_reg, b, w, train_acc)
            print(f"Epoch {epoch:4d} | Loss: {loss:.5f} | Acc: {train_acc:.4f} | |w|_std: {np.std(w):.4f}")
            
    # Финальная запись и сохранение параметров
    log_step(epochs, loss, lr, lambda_reg, b, w, accuracy_score(y, (y_pred >= 0.5).astype(int)))
    with open(PARAMS_FILE, 'w') as f:
        json.dump({"w": w.tolist(), "b": float(b), "lambda_reg": lambda_reg, "epochs": epochs}, f, indent=2)
    print(f"\n✅ Обучение завершено. Логи сохранены в `{LOG_FILE}`, параметры в `{PARAMS_FILE}`")
    return w, b, loss_history

## 🎯 Предсказание и метрики
- **Вероятности:** $\hat{y} = \sigma(\mathbf{w}^T\mathbf{x} + b)$
- **Класс:** $y_{pred} = \mathbb{I}(\hat{y} \geq 0.5)$
- **Accuracy:** $\frac{1}{m} \sum_{i=1}^{m} \mathbb{I}(y^{(i)} = y_{pred}^{(i)})$
- **Confusion Matrix:** TP, FP, TN, FN для оценки false-positive/false-negative ошибок.

In [6]:
def predict_proba(X, w, b):
    return sigmoid(np.dot(X, w) + b)

def predict(X, w, b, threshold=0.5):
    return (predict_proba(X, w, b) >= threshold).astype(int)

def evaluate_model(X, y, w, b, threshold=0.5):
    y_pred = predict(X, w, b, threshold)
    acc = accuracy_score(y, y_pred)
    report = classification_report(y, y_pred, target_names=['Не выжил', 'Выжил'], output_dict=True)
    cm = confusion_matrix(y, y_pred)
    return acc, report, cm, y_pred

In [ ]:
# 1. Данные
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_titanic()

# 2. Обучение (можно менять lr, epochs, lambda_reg)
w, b, loss_hist = train_logistic_regression(
    X_train, y_train, 
    lr=0.5, epochs=1500, lambda_reg=0.01, log_interval=100
)

# 3. Оценка на тесте
acc, report, cm, y_test_pred = evaluate_model(X_test, y_test, w, b)
print(f"\n📊 Accuracy на тесте: {acc:.4f}")
print("\n📋 Classification Report:")
print(pd.DataFrame(report).T.round(4))

# 4. Визуализация
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(loss_hist, color='purple', linewidth=2)
axes[0].set_title("Динамика функции потерь")
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("BCE Loss")
axes[0].grid(True, alpha=0.3)

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['0', '1'], yticklabels=['0', '1'], ax=axes[1])
axes[1].set_title("Confusion Matrix (Test)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 🔍 Анализ файла `training_log.csv`
После обучения в папке появится файл с полной историей. Его можно загрузить и проанализировать:


In [7]:
log_df = pd.read_csv('training_log.csv')
log_df.plot(x='epoch', y=['loss', 'train_acc'], secondary_y='train_acc', figsize=(8,4))
plt.title("Прогресс обучения из лога")
plt.grid(True, alpha=0.3)
plt.show()

NameError: name 'pd' is not defined